# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiments: **E005a-mm-crop + E005b-perlabel-attention, one 2x2 run**
on the production resnet34 @224 arm. Two per-slice banks (full-frame and a fixed
140mm crop — two decode passes) x two head types (`mean_max`, `attention`), four
CVs off identical folds. The full_frame/mean_max cell is the anchor: it must
reproduce E003's ~0.771 or nothing else counts. Each margin of the square
attributes one lever; the fourth cell measures their interaction. Requires the
`WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E005a/E005b — bump to the mm-crop squash merge before `kaggle kernels push`
# (this run needs crop_mm plumbing plus HeadType/PerLabelAttentionHead, absent at b8d29f8).
COMMIT = "b8d29f8"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE, train_heads_from_bank

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; the blended soft
# labels arrive via the attached private knee-labels dataset (issue #2 resolved).
from pathlib import Path

from knee.data import load_blended_labels

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

# Attached datasets moved mount points too; probe the plausible locations.
LABELS_CSV = "blended_labels_v1.csv"
label_candidates = [
    Path("/kaggle/input/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/josiemachalek/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/knee-labels") / LABELS_CSV,
]
labels_path = next((p for p in label_candidates if p.exists()), None)
if labels_path is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{LABELS_CSV} not found; mounts: {listing}")
labels = load_blended_labels(labels_path)
print(f"blended labels: {len(labels)} studies from {labels_path}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E005a x E005b: production backbone; the levers are the fixed-mm crop (140mm from
# the forum's measured ablations, docs/rsna_brain.md §2.35-2.36; median corpus FOV
# is 160mm) and the pooling head. Banks are per-slice, so head choice is fit-time.
from knee.model import DEFAULT_BACKBONE, HeadType, KneeModel

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224
BANKS = {"full_frame": None, "crop140": 140.0}  # tag -> crop_mm
ARMS = (HeadType.MEAN_MAX, HeadType.ATTENTION)  # control first

CHECKPOINT_DIR = Path("/kaggle/working")

def bank_path(tag: str) -> Path:
    return CHECKPOINT_DIR / f"slice_bank_{BLENDED_LABEL_SOURCE}_resnet34_{tag}.pt"

In [ ]:
# Per bank: one threaded decode pass (the hours; two banks = two passes) then per
# head type a fit from cache (minutes). One shared extractor backbone for all four
# cells — the head-carrying models clone its weights so every checkpoint pairs a
# head with the exact backbone that produced its training features. Checkpoints
# land in per-cell subdirs (e.g. crop140/attention/); crop_mm is stamped into each
# checkpoint so inference reproduces the same physical window.
from knee.cv import collect_features, save_feature_bank

extractor = KneeModel(BACKBONE)
banks, cell_results = {}, {}
for tag, crop_mm in BANKS.items():
    print(f"=== bank {tag} (crop_mm={crop_mm}) ===")
    bank = collect_features(
        COMP_ROOT, labels, series_types=SERIES_TYPES, model=extractor,
        input_size=INPUT_SIZE, crop_mm=crop_mm,
    )
    save_feature_bank(bank, bank_path(tag))
    banks[tag] = bank
    print("plane coverage:", {t.value: n for t, n in bank.plane_coverage().items()})
    for head_type in ARMS:
        if head_type is extractor.head_type:
            carrier = extractor
        else:
            carrier = KneeModel(BACKBONE, head_type=head_type)
            carrier.backbone.load_state_dict(extractor.backbone.state_dict())
        results = train_heads_from_bank(
            bank, CHECKPOINT_DIR / tag / head_type.value, carrier,
            input_size=INPUT_SIZE, crop_mm=crop_mm,
        )
        cell_results[(tag, head_type)] = results
        for result in results:
            print(f"{tag}/{head_type.value}/{result.series_type.value}: trained on {result.n_studies} studies")

In [ ]:
# Local eval: pooled-OOF stratified CV per 2x2 cell, all four from identical
# folds/seed — margins attribute the levers, the diagonal shows interaction.
# Sanity gate: full_frame/mean_max must land at ~0.771 (E003) or the bank refactor
# changed something and nothing else counts. Decision rule (experiments.md
# E005a/E005b): submit the best cell only if it beats the anchor by more than the
# per-repeat spread — E004 showed CV gains may not transfer, and a null isn't
# worth a submission.
from knee.cv import cross_validate

cvs = {}
for tag in BANKS:
    for head_type in ARMS:
        print(f"=== {tag}/{head_type.value} ===")
        cvs[(tag, head_type)] = cv = cross_validate(banks[tag], head_type=head_type)
        print(f"macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
              + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
        print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints land in per-cell subdirs (<bank>/<head>/) plus both banks, all
# persisted as notebook output. If the decision rule passes, publish the winning
# cell's three .pt files as a new knee-weights dataset VERSION that REPLACES the
# previous ones (inference rejects duplicate series types — never mount two sets);
# crop_mm rides inside each checkpoint, so inference needs no config.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE,
        "banks": {tag: crop for tag, crop in BANKS.items()},
        "arms": [a.value for a in ARMS],
    })
    for (tag, head_type), results in cell_results.items():
        wandb.log(
            {
                f"in_sample_auc/{tag}/{head_type.value}/{result.series_type.value}/{label}": auc
                for result in results
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    for (tag, head_type), cv in cvs.items():
        wandb.log(
            {
                f"cv/macro_auc/{tag}/{head_type.value}": cv.macro_auc,
                **{f"cv/auc/{tag}/{head_type.value}/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
            }
        )
    run.finish()